In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sys
import os


feature_path = os.path.abspath(r"C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor")
# feature_path = os.path.abspath('/Users/alexg/Documents/Documents/Prize-Picks-Prop-Predictor')
if feature_path not in sys.path:
    sys.path.append(feature_path)

from FEATURE_ENGINEERING.features import *
from MODELS.pipeline import *
from PROPS_EV.calculateEVS import *

### Load Model

In [2]:
model = joblib.load('Models/xgbModel.pkl')
features = joblib.load('Models/top_features.pkl')

### Load Data

In [62]:
pd.set_option('display.max_columns', None)
eplison = 0.000001

data = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_25.csv')
data['EXPECTED_USAGE_MIN'] = data['USG_PCT_ROLLING_AVG_5'] * (data['MIN_ROLLING_AVG_5'] + eplison)

data2 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_24.csv')
data2['EXPECTED_USAGE_MIN'] = data2['USG_PCT_ROLLING_AVG_5'] * (data2['MIN_ROLLING_AVG_5'] + eplison)
df = pd.concat([data, data2]).sort_values(by='GAME_DATE')


date = '2024-10-24'
backtestData = pd.read_csv('../DATA/CSV_FILES/BACKTEST_DATA/singleBookies.csv')
backtestData

,NAME,CATEGORY,SIDE,BOOKMAKER,LINE,ODDS,fair_line,fair_odds,GAME_DATE
0,Luke Kornet,assists,over,draftkings,1.5,205,1.5,226,2024-10-22
1,Josh Hart,rebounds,over,fanduel,2.5,-122,4.5,100,2024-10-22
2,Sam Hauser,assists,over,draftkings,1.5,215,1.5,236,2024-10-22
3,Josh Hart,rebounds,under,fanduel,2.5,-108,4.5,100,2024-10-22
4,Josh Hart,assists,under,fanduel,2.5,152,4.0,100,2024-10-22
...,...,...,...,...,...,...,...,...,...
234574,Jeff Green,points,under,espnbet,1.5,100,1.5,105,2025-04-11
234575,Jeff Green,rebounds,over,espnbet,3.5,135,3.5,126,2025-04-11
234576,Jeff Green,rebounds,under,espnbet,3.5,-190,3.5,-126,2025-04-11
234577,Jeff Green,rebounds,over,espnbet,0.5,-170,0.5,-142,2025-04-11


In [ ]:
# results = single_bet(
#     data=df,
#     bookmakers=backtestData,
#     model=model,
#     features=features,
#     stake=100,
#     simulations=10000
# )
# evData = results.sort_values(by='EV%', ascending=False).head(10)
# evData

Processing single bets...


,NAME,BOOKMAKER,CATEGORY,LINE,ODDS,SIDE,PREDICTION,OVER%,UNDER%,IMPLIED PROB,EV%,KELLY FULL,KELLY HALF,KELLY QUARTER,CONFIDENCE INTERVAL
1,Jrue Holiday,betrivers,points,13.5,-112,under,9.954951,0.130,0.870,0.528,64.66,0.72,0.36,0.18,"(3.8, 16.2)"
11,Jayson Tatum,betrivers,points,30.5,-124,under,22.858770,0.110,0.890,0.554,60.79,0.75,0.38,0.19,"(10.8, 35.1)"
14,Kyle Kuzma,betrivers,points,19.5,-114,under,13.312409,0.162,0.838,0.533,57.31,0.65,0.33,0.16,"(2.4, 25.8)"
18,Luka Dončić,fanduel,points,29.5,250,over,27.763483,0.432,0.568,0.286,51.16,0.20,0.10,0.05,"(10.5, 45.2)"
13,Corey Kispert,betrivers,points,11.5,-125,under,6.096945,0.201,0.799,0.556,43.77,0.55,0.27,0.14,"(0.5, 17.5)"
12,Jordan Poole,betrivers,points,17.5,-113,under,14.494495,0.285,0.715,0.531,34.85,0.39,0.20,0.10,"(4.2, 25.1)"
0,Jaylen Brown,betrivers,points,22.5,-130,under,19.265526,0.342,0.658,0.565,16.36,0.21,0.11,0.05,"(4.3, 35.0)"
17,Bilal Coulibaly,betrivers,points,8.5,-110,over,9.716388,0.607,0.393,0.524,15.81,0.17,0.09,0.04,"(1.3, 20.5)"
4,Al Horford,betrivers,points,9.5,-109,over,10.414342,0.591,0.409,0.522,13.22,0.14,0.07,0.04,"(3.0, 18.3)"
2,Derrick White,betrivers,points,15.5,-121,over,15.638899,0.519,0.481,0.548,-5.17,0.00,0.00,0.00,"(2.1, 33.2)"


In [72]:
def backtest(data, backtestData, gameDate):
    data = data[data['GAME_DATE'] <= gameDate]
    backtestData = backtestData[(backtestData['CATEGORY'] == 'points') & (backtestData['GAME_DATE'] == gameDate)]
    if backtestData.empty:
        print(f"No bets found for {gameDate}")
        return pd.DataFrame()

    results = single_bet(
    data=data,
    bookmakers=backtestData,
    model=model,
    features=features,
    stake=100,
    simulations=10000)
    evData = results.sort_values(by='EV%', ascending=False).head(10)
    results = []

    for idx, row in evData.iterrows():
        player = row['NAME']
        bookmaker = row['BOOKMAKER']
        side = row['SIDE']
        line = row['LINE']
        odds = row['ODDS']
        pred = row['PREDICTION']
        
        playerData = data[data['PLAYER_NAME'] == player]

        if playerData.empty:
            print(f"Player {player} not found in data for {gameDate}")
            continue

        actual = playerData['PTS'].iloc[-1]
        edge = pred - line
        
        if side == 'over':
            recommendation = 1 if (pred > line and abs(edge) > 4) else 0
        elif side == 'under':
            recommendation = 1 if (pred < line and abs(edge) > 4) else 0
        else:
            recommendation = 0

        if side == 'over':
            won = 1 if actual > line else 0
        elif side == 'under':
            won = 1 if actual < line else 0
        else:
            won = 0

        results.append({
            'player': player, 
            'bookmaker': bookmaker, 
            'side': side, 
            'line': line, 
            'odds': odds, 
            'pred': pred, 
            'actual': actual,
            'edge': edge,
            'recommendation': recommendation,
            'won': won,
        })

    return pd.DataFrame(results)

results = backtest(df, backtestData, '2025-03-27')

# Calculate metrics
if not results.empty:
    # Overall metrics
    total_bets = len(results)
    total_wins = results['won'].sum()
    win_rate = results['won'].mean()
    
    # Metrics for recommended bets only
    recommended = results[results['recommendation'] == 1]
    if not recommended.empty:
        rec_total = len(recommended)
        rec_wins = recommended['won'].sum()
        rec_win_rate = recommended['won'].mean()
        
        print(f"\n--- All Bets ---")
        print(f"Total: {total_bets}, Wins: {total_wins}, Win Rate: {win_rate:.2%}")
        print(f"\n--- Recommended Bets (Edge > 4) ---")
        print(f"Total: {rec_total}, Wins: {rec_wins}, Win Rate: {rec_win_rate:.2%}")
results

Processing single bets...

--- All Bets ---
Total: 10, Wins: 5, Win Rate: 50.00%

--- Recommended Bets (Edge > 4) ---
Total: 8, Wins: 4, Win Rate: 50.00%


,player,bookmaker,side,line,odds,pred,actual,edge,recommendation,won
0,Fred VanVleet,fanduel,over,4.5,178,15.885443,4,11.385443,1,0
1,Andrew Nembhard,fanduel,over,7.5,188,12.413404,7,4.913404,1,0
2,Johnny Juzang,fanduel,over,11.5,320,12.359076,11,0.859076,0,0
3,Jarred Vanderbilt,fanduel,over,4.5,188,6.252589,7,1.752589,0,1
4,Patrick Williams,draftkings,over,8.5,160,13.379631,11,4.879631,1,1
5,Franz Wagner,fanduel,over,16.5,154,22.846664,20,6.346664,1,1
6,Naji Marshall,espnbet,over,6.5,125,21.197983,6,14.697983,1,0
7,Naji Marshall,fanduel,over,4.5,124,21.197983,6,16.697983,1,1
8,Jarred Vanderbilt,espnbet,over,0.5,120,6.252589,7,5.752589,1,1
9,Donovan Clingan,espnbet,over,2.5,120,8.120650,0,5.620650,1,0


ideas with backtest:
- test with different edge thresholds